# Answer Key

One worked answer per TODO cell. Cell IDs match the notebooks.

## 01 Triage the Queue

**`todo-query`**

```python
query = alerts.query_alerts_v2(limit=10, filter="product:'epp'+tactic:'Privilege Escalation'")
```

## 02 Scope the Host

**`todo-carry`**

```python
trigger_aid = "<agent_id from notebook 01>"
```

**`todo-scope`**

```python
scope = alerts.query_alerts_v2(limit=20, filter=f"agent_id:'{trigger_aid}'")
```

**`todo-vuln`**

```python
vuln_resp = vulns.query_vulnerabilities_combined(
    limit=20,
    filter=f"aid:'{trigger_aid}'+status:'open'+cve.is_cisa_kev:true",
    facet=["cve", "remediation"],
)
```

**`todo-attr`**

```python
actors = vuln["cve"].get("actors")
```

**`todo-pag`**

```python
def fetch_exploited_highs():
    found = []
    after = None

    while True:
        page = vulns.query_vulnerabilities(
            limit=50,
            filter="status:'open'+cve.severity:'HIGH'+cve.exploit_status:'90'",
            after=after,
        )
        found.extend(page["body"]["resources"])
        after = page["body"]["meta"]["pagination"].get("after")
        if not after:
            break

    return found
```

## 03 Assess Exposure

**`todo-carry`**

```python
cve_id = "<cve_id from notebook 02>"
```

**`todo-exp`**

```python
exposure = vulns.query_vulnerabilities_combined(
    limit=100, filter=f"cve.id:'{cve_id}'+status:'open'", facet=["cve", "host_info"]
)
```

## 04 Get Remediation

**`todo-carry`**

```python
cve_id = "<cve_id from notebook 02 or 03>"
```

**`todo-res`**

```python
resp = vulns.get_remediations_v2(ids=[remediation_id])
```

## 05 Close the Case

**`todo-carry`**

```python
trigger_aid = "<agent_id from notebook 01>"
trigger_summary = "Privilege Escalation attempt"
cve_id = "<root-cause cve_id from notebook 02>"
case_actors = [...]        # from notebook 02
exposed_host_count = 0     # from notebook 03
fix_action = "<action text from notebook 04>"
```

## Notes

- `get_alerts_v2` takes `composite_ids=`, not `ids=`.
- `query_vulnerabilities_combined` has no server-side filter on app or product name.
  `cve.is_cisa_kev:true` narrows the set to something matchable by hand.
- Spotlight only correlates against package-manager metadata, so the `bash` and `cat` stages in
  the alert history have no CVE.
- Spotlight pages with an opaque `after` cursor. Max `limit` is 400.
- If notebook 01 returns nothing, the seeded data may not be loaded for this tenant. Every
  notebook has a fallback cell that loads a saved response from `data/sample_responses/`.

## 07 Exercises

**Exercise 1: Hosts**

```python
response = hosts.query_devices_by_filter(filter="platform_name:'Linux'")
devices = hosts.get_device_details(ids=response["body"]["resources"])["body"]["resources"]

for device in devices:
    print(device["hostname"], "|", device["os_version"], "|", device["local_ip"])
```

**Exercise 2: Alerts**

```python
trigger_aid = "ffd14c2909644380bd0cf9e8ce471c41"

resp = alerts.query_alerts_v2(filter=f"agent_id:'{trigger_aid}'+product:'epp'+tactic:'Command and Control'")
c2_alerts = alerts.get_alerts_v2(composite_ids=resp["body"]["resources"])["body"]["resources"]

for alert in c2_alerts:
    print(alert.get("cmdline"))
```

**Exercise 3: Spotlight Vulnerabilities**

```python
exposure = vulns.query_vulnerabilities_combined(
    filter="cve.id:'CVE-2021-4034'+status:'open'",
    facet=["cve", "host_info"]
)

print("hosts exposed:", exposure["body"]["meta"]["pagination"]["total"])
for vuln in exposure["body"]["resources"]:
    print(vuln["host_info"]["hostname"], "|", ", ".join(vuln["cve"].get("actors") or []))
```

**Exercise 4: Discover**

```python
trigger_aid = "ffd14c2909644380bd0cf9e8ce471c41"

resp = discover.query_applications(filter=f"host.aid:'{trigger_aid}'", limit=100)
print("total apps:", resp["body"]["meta"]["pagination"]["total"])

apps = discover.get_applications(ids=resp["body"]["resources"])["body"]["resources"]
for app in apps:
    if "git" in app.get("name", "").lower() or "policykit" in app.get("name", "").lower():
        print(app["name"], "|", app.get("version"))
```

**Exercise 5: Intel**

```python
resp = intel.query_actor_entities(filter="slug:'labyrinth-chollima'")
actor = resp["body"]["resources"][0]

print("name:", actor["name"])
print("origins:", [o["value"] for o in actor.get("origins", [])])
print("motivations:", [m["value"] for m in actor.get("motivations", [])])
print("targets:", [t["value"] for t in actor.get("target_industries", [])])
```